In [ ]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate
from openai import OpenAI

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_json(PROJECT_ROOT / "datasets" / "ViNumQA" / "test.json")
df.sample(n=5)

In [ ]:
len(df)

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

df["pre_text_processed"] = df.apply(lambda x: formatting_pre_text(x), axis=1)
df["post_text_processed"] = df.apply(lambda x: formatting_post_text(x), axis=1)
df["table_processed"] = df.apply(lambda x: formatting_table(x), axis=1)
df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
df["input_question"] = df.apply(lambda x: processing_input_question(x), axis=1)
df["program_processed"] = df.apply(lambda x: processing_program_content(x), axis=1)
df["answer_processed"] = df.apply(lambda x: processing_answer_content(x), axis=1)
df.sample(n=5)

In [ ]:
df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question", "program_processed", "answer_processed"]]
df.columns = [["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]]
df["generated_program"] = ""
df["calculated_program"] = ""
df

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["API_KEY"]
BASE_URL = os.environ["BASE_URL"]

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

MODEL = "Llama-3.3-70B-Instruct" # gemma-3-27b-it, Llama-3.3-70B-Instruct, DeepSeek-V4-Flash

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}
 
[TABLE]
{table}
 
[TEXT AFTER TABLE]
{post_text}
 
### QUESTION:
{question}
 
### PROGRAM:"""

# 3 few-shot demonstrations sampled from train.json, one per evidence type
# (Table Only / Text Only / Table & Text), following the ViNumQA task's
# own categorization (see VLSP 2025 NumQA paper, Section 3.3).
# The "table" field of each shot is pre-rendered to the same GitHub-markdown
# format that `formatting_table` produces for the real data, so the few-shot
# demonstrations are formatted identically to the actual queries.
FEW_SHOT_EXAMPLES = [
    {
        # Table Only (train idx 1959): answer is derived purely from two table cells.
        "pre_text": "phụ lục iv ace limited và các công ty con thông tin bổ sung về phí tái bảo hiểm thu được cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la mỹ, ngoại trừ tỷ lệ phần trăm) số tiền trực tiếp nhượng cho các công ty nhận từ các công ty khác số tiền ròng tỷ lệ phần trăm số tiền nhận được trên.",
        "table": (
            "|   cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la Mỹ, ngoại trừ tỷ lệ phần trăm) | số tiền trực tiếp   | nhượng cho các công ty khác   | nhận từ các công ty khác   | số tiền ròng   | tỷ lệ phần trăm số tiền nhận được trên số tiền ròng   |\n"
            "|----------------------------------------------------------------------------------------------------------------------|---------------------|-------------------------------|----------------------------|----------------|-------------------------------------------------------|\n"
            "|                                                                                                                 2010 | $ 15780             | $ 5792                        | $ 3516                     | $ 13504        | 26% ( 26 % )                                          |\n"
            "|                                                                                                                 2009 | $ 15415             | $ 5943                        | $ 3768                     | $ 13240        | 28% ( 28 % )                                          |\n"
            "|                                                                                                                 2008 | $ 16087             | $ 6144                        | $ 3260                     | $ 13203        | 25% ( 25 % )                                          |"
        ),
        "post_text": ".",
        "question": "Sự khác biệt giữa số tiền chuyển giao và nhận chuyển giao trong năm 2010 là bao nhiêu?",
        "program": "subtract(5792, 3516)",
    },
    {
        # Text Only (train idx 93): the supplied table (VHM financial summary) is
        # irrelevant to the question; the program only uses numbers from pre_text.
        "pre_text": "hệ số khả năng thanh toán lãi vay cũng tăng cao đạt mức 13.2 lần, so với chỉ 10.3 lần cùng kỳ.",
        "table": (
            "|                   |   FY 2015 |   FY 2016 |   FY 2017 |   FY 2018 |   FY 2019(F) |\n"
            "|-------------------|-----------|-----------|-----------|-----------|--------------|\n"
            "| Doanh thu (VNDbn) |      4920 |     11217 |     15297 |     38664 |        71115 |\n"
            "| Lãi gộp (Vbn)     |       718 |      2420 |      3128 |      7617 |        10983 |"
        ),
        "post_text": ".",
        "question": "Hệ số khả năng thanh toán lãi vay tăng bao nhiêu lần so với cùng kỳ năm ngoái?",
        "program": "subtract(13.2, 10.3)",
    },
    {
        # Table & Text (train idx 1127): must locate the right table row ("Nội dung số")
        # across three columns and chain two operators via the #0 reference.
        "pre_text": "tỷ lệ phần trăm chi phí vốn trên phần trăm tổng tài sản của mảng viễn thông được duy trì trên 1, cho thấy sự tập trung phân bổ chi phí vốn vào mảng viễn thông của fpt qua các năm.\nngoài ra, tỷ lệ này của mảng đầu tư và giáo dục là 1,1 vào năm 2018 và 0,9 vào năm 2019, khẳng định fpt cũng đang tập trung vào phát triển 2 mảng này trong 2 năm gần đây.",
        "table": (
            "|                     |   2014 |   2015 |   2016 |   2017 |   2018 |   2019 |\n"
            "|---------------------|--------|--------|--------|--------|--------|--------|\n"
            "| Viễn thông          |    1.8 |    2.4 |    1.9 |    1.4 |    1.7 |    1.8 |\n"
            "| Nội dung số         |    0.4 |    0.2 |    0.9 |    0.1 |    0.1 |    0.1 |\n"
            "| Phát triển phần mềm |    2.4 |    1.3 |    3.1 |    1.1 |    0.4 |    0.5 |"
        ),
        "post_text": ".",
        "question": "Tổng tỷ lệ của mảng Nội dung số trong ba năm từ 2014 đến 2016 là bao nhiêu?",
        "program": "add(0.4, 0.2), add(#0, 0.9)",
    },
]

In [ ]:
from tqdm import tqdm

# Build the fixed few-shot prefix once: alternating user/assistant turns,
# one pair per FEW_SHOT_EXAMPLES entry, each following the same USER_MESSAGE_FRAME
# used for the real query so the model sees a consistent input/output format.
few_shot_messages = []
for shot in FEW_SHOT_EXAMPLES:
    few_shot_messages.append({
        "role": "user",
        "content": USER_MESSAGE_FRAME.format(
            pre_text=shot["pre_text"],
            table=shot["table"],
            post_text=shot["post_text"],
            question=shot["question"],
        )
    })
    few_shot_messages.append({"role": "assistant", "content": shot["program"]})

for df_index, values in tqdm(df.iterrows(), total=len(df), desc="Generating program..."):
    pre_text = values["pre_text"]
    table = values["table"]
    post_text = values["post_text"]
    question = values["question"]

    chat_completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_MESSAGE,
            },
            *few_shot_messages,
            {
                "role": "user",
                "content": USER_MESSAGE_FRAME.format(pre_text=pre_text, table=table, post_text=post_text, question=question)
            },
        ],
            temperature=0.0,
            max_tokens=8096,
            stream=False
            )
    output = chat_completion.choices[0].message.content.strip().strip("\n")

    df.at[df_index, "generated_program"] = output

    print(f"TEST SAMPLE {df_index}:\n\nPREDICTION:\n{output}\n\nGROUND_TRUTH:\n{values['program']}")
    print("="*100)

In [ ]:
import sys
sys.path.insert(0, "../../evaluate")  # fallback: relative path when running locally

from scorer import evaluate_dataframe  # noqa: E402

# scorer.py is the shared ViNumQA evaluator (notebooks/evaluate/scorer.py): it
# ports FinQA's official evaluation protocol (sympy-based symbolic Program
# Accuracy, table-row-lookup-aware Execution Accuracy) instead of a
# hand-rolled parser, and correctly executes table_*(row_name, none) calls by
# looking up the named row in the raw table -- which the previous in-notebook
# parser could not do at all (it treated table_* arguments as raw numbers).

In [ ]:
df_scored, summary = evaluate_dataframe(
    df,
    generated_col="generated_program",   # cột bạn đang ghi output model vào
    gold_program_col="program",
    gold_answer_col="answer",
    table_col="table_raw",
)

print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}

In [12]:
import json
from pathlib import Path

results_path = Path(f"outputs/few_shot_{MODEL}_results.csv")
summary_path = Path(f"outputs/few_shot_{MODEL}_summary.json")
results_path.parent.mkdir(parents=True, exist_ok=True)

df_scored.to_csv(results_path, index=False)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "shot": "few-shot", "max_tokens": 8096, **summary}, f, ensure_ascii=False, indent=2)

print(f"Saved per-sample results to {results_path}")
print(f"Saved summary to {summary_path}")

Saved per-sample results to outputs\few_shot_Llama-3.3-70B-Instruct_results.csv
Saved summary to outputs\few_shot_Llama-3.3-70B-Instruct_summary.json
